# UAV Battery Tool — Notebook 00: Bulk Data Entry

Add new batteries and equipment to the catalogue, or scrape data from websites and PDF datasheets.

**Sections:**
1. Add a new battery pack
2. Add new equipment
3. Scrape battery data from a URL
4. Extract data from a PDF datasheet

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import re
import pandas as pd

from batteries.database import BatteryDatabase
from batteries.models import BatteryPack, Equipment

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

═══ Battery Database Summary ═══
  Chemistries       : 9
  Cells             : 11
  Battery packs     : 8
  Discharge points  : 132
  Equipment items   : 29
  UAV configurations: 3
  Mission profiles  : 3


---
## 1 · Add a New Battery Pack

Fill in the fields below and run the cell to append the pack to `battery_db.xlsx`.

In [2]:
# Preview existing chemistries and cells to help you fill in the form
print('Available chemistry IDs:')
for cid, chem in db.chemistries.items():
    print(f'  {cid:<12} {chem.name}')
print()
print('Available cell IDs:')
for cid, cell in db.cells.items():
    print(f'  {cid:<30} {cell.chemistry_id:<10} {cell.capacity_ah}Ah  {cell.internal_resistance_mohm}mΩ')

Available chemistry IDs:
  LIPO         Lithium Polymer
  LION         Lithium Ion (NMC)
  LION21       Lithium Ion 21700 (NMC)
  LIFEPO4      Lithium Iron Phosphate
  LITO         Lithium Titanate
  SSS          Semi-Solid State (Li-Ion based)
  SOLID        All-Solid-State
  NIMH         Nickel Metal Hydride
  LIHV         High-Voltage LiPo (LiHV)

Available cell IDs:
  TATTU_1300_4S                  LIPO       1.3Ah  15.0mΩ
  GAONENG_550                    LIHV       0.55Ah  30.0mΩ
  CNHL_5200                      LIPO       5.2Ah  10.0mΩ
  SAMSUNG_30Q                    LION       3.0Ah  28.0mΩ
  SAMSUNG_40T                    LION21     4.0Ah  18.0mΩ
  MOLICEL_P45B                   LION21     4.5Ah  14.0mΩ
  LG_M50LT                       LION21     5.0Ah  22.0mΩ
  A123_ANR26650                  LIFEPO4    2.5Ah  8.0mΩ
  HEADWAY_40152                  LIFEPO4    10.0Ah  5.0mΩ
  24M_SERIES                     SSS        5.0Ah  20.0mΩ
  TOYOTA_SSB_REF                 SOLID      3.0

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  BATTERY PACK FORM — fill in every field, then run this cell
# ══════════════════════════════════════════════════════════════════════════════

NEW_PACK = dict(
    battery_id             = 'BAT_MY_PACK_01',     # unique ID — no spaces
    name                   = 'My Custom 6S3P Pack', # human-readable name
    cell_id                = 'LG_M50LT',            # must exist in Cell_Catalog (see Section 1 output above)
    chemistry_id           = 'LION21',              # must exist in Chemistry_Library
    cells_series           = 6,                     # S in SxP config
    cells_parallel         = 3,                     # P in SxP config

    # ── Electrical ──────────────────────────────────────────────────────────
    # Leave as None to auto-calculate from cell data + S/P config
    pack_voltage_nom       = None,   # V  (auto = cell_voltage_nom × S)
    pack_voltage_max       = None,   # V  (auto = cell_voltage_max × S)
    pack_voltage_cutoff    = None,   # V  (auto = cell_voltage_cutoff × S)
    pack_capacity_ah       = None,   # Ah (auto = cell_capacity_ah × P)
    pack_energy_wh         = None,   # Wh (auto = voltage_nom × capacity_ah)
    internal_resistance_mohm = None, # mΩ (auto = cell_ir × S / P)
    max_cont_discharge_a   = None,   # A  (auto = cell_max_cont × P)
    max_cont_discharge_w   = None,   # W  (auto = max_a × voltage_nom)

    # ── Physical ────────────────────────────────────────────────────────────
    pack_weight_g          = None,   # g  (auto = cell_weight × S × P × 1.15 overhead)
    uav_class              = 'Survey',  # e.g. 'Survey', 'Racing', 'Heavy-lift'
    notes                  = '',
)

# ══════════════════════════════════════════════════════════════════════════════

# Auto-fill None fields from cell data
if NEW_PACK['cell_id'] in db.cells:
    cell = db.cells[NEW_PACK['cell_id']]
    S, P = NEW_PACK['cells_series'], NEW_PACK['cells_parallel']
    def _auto(key, val):
        return NEW_PACK[key] if NEW_PACK[key] is not None else val
    v_nom  = _auto('pack_voltage_nom',       round(cell.voltage_nominal * S, 2))
    v_max  = _auto('pack_voltage_max',        round(cell.voltage_max * S, 2))
    v_cut  = _auto('pack_voltage_cutoff',     round(cell.voltage_cutoff * S, 2))
    cap_ah = _auto('pack_capacity_ah',        round(cell.capacity_ah * P, 3))
    e_wh   = _auto('pack_energy_wh',          round(v_nom * cap_ah, 1))
    ir     = _auto('internal_resistance_mohm',round(cell.internal_resistance_mohm * S / P, 2))
    i_max  = _auto('max_cont_discharge_a',    round(cell.max_cont_discharge_a * P, 1))
    w_max  = _auto('max_cont_discharge_w',    round(i_max * v_nom, 0))
    wt     = _auto('pack_weight_g',           round(cell.weight_g * S * P * 1.15, 0))
else:
    avail = list(db.cells.keys())
    raise ValueError(
        f'cell_id "{NEW_PACK["cell_id"]}" not found in Cell_Catalog.\n'
        f'Available cell IDs: {avail}\n'
        'Update the cell_id field above to one of these.'
    )

print('Pack preview (auto-calculated fields shown with *):')
print(f'  ID                   : {NEW_PACK["battery_id"]}')
print(f'  Name                 : {NEW_PACK["name"]}')
print(f'  Config               : {S}S{P}P ({S*P} cells total)')
print(f'  Chemistry            : {NEW_PACK["chemistry_id"]}')
print(f'  Voltage nominal      : {v_nom} V')
print(f'  Voltage max          : {v_max} V')
print(f'  Voltage cutoff       : {v_cut} V')
print(f'  Capacity             : {cap_ah} Ah')
print(f'  Energy               : {e_wh} Wh')
print(f'  Internal resistance  : {ir} mΩ')
print(f'  Max cont. discharge  : {i_max} A  /  {w_max} W')
print(f'  Estimated weight     : {wt} g')

Pack preview (auto-calculated fields shown with *):
  ID                   : BAT_MY_PACK_01
  Name                 : My Custom 6S3P Pack
  Config               : 6S3P (18 cells total)
  Chemistry            : LION21
  Voltage nominal      : 21.78 V
  Voltage max          : 25.2 V
  Voltage cutoff       : 15.0 V
  Capacity             : 15.0 Ah
  Energy               : 326.7 Wh
  Internal resistance  : 44.0 mΩ
  Max cont. discharge  : 42.0 A  /  915.0 W
  Estimated weight     : 1459.0 g


In [4]:
# ── Set SAVE_PACK = True to write to battery_db.xlsx ─────────────────────────
SAVE_PACK = False

if SAVE_PACK:
    if NEW_PACK['battery_id'] in db.packs:
        print(f'Warning: {NEW_PACK["battery_id"]} already exists — will overwrite row.')
    pack_obj = BatteryPack(
        battery_id=NEW_PACK['battery_id'], name=NEW_PACK['name'],
        cell_id=NEW_PACK['cell_id'], chemistry_id=NEW_PACK['chemistry_id'],
        cells_series=S, cells_parallel=P,
        pack_voltage_nom=v_nom, pack_voltage_max=v_max, pack_voltage_cutoff=v_cut,
        pack_capacity_ah=cap_ah, pack_energy_wh=e_wh,
        pack_weight_g=wt,
        internal_resistance_mohm=ir,
        max_cont_discharge_a=i_max, max_cont_discharge_w=w_max,
        uav_class=NEW_PACK['uav_class'], notes=NEW_PACK['notes'],
    )
    db.append_custom_pack(pack_obj)
    print(f'Saved {NEW_PACK["battery_id"]} to {DB_PATH}')
else:
    print('Preview only — set SAVE_PACK = True to write to the database.')

Preview only — set SAVE_PACK = True to write to the database.


---
## 2 · Add New Equipment

Fill in the power figures for each flight phase, then save.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  EQUIPMENT FORM — fill in every field, then run this cell
# ══════════════════════════════════════════════════════════════════════════════

NEW_EQUIP = dict(
    equip_id        = 'MOT_MY_MOTOR_KV400',  # unique ID — no spaces
    category        = 'Propulsion',           # Propulsion / Avionics / Payload / Comms
    manufacturer    = 'T-Motor',
    model           = 'MN601-S KV400',
    nom_voltage_v   = 22.2,                   # nominal operating voltage
    nom_current_a   = 20.0,                   # nominal current at hover

    # ── Power per phase (W per unit — multiply by qty in UAV config) ─────────
    # Use manufacturer data or thrust-stand measurements.
    idle_power_w    = 5.0,     # armed but not flying
    hover_power_w   = 180.0,   # sustained hover
    climb_power_w   = 230.0,   # full climb (~110-125% of hover)
    cruise_power_w  = 130.0,   # forward cruise (~70-80% of hover)
    max_power_w     = 400.0,   # full throttle burst

    # ── Physical ────────────────────────────────────────────────────────────
    weight_g        = 185.0,   # motor + ESC combined weight
    efficiency_pct  = 85.0,    # motor+ESC combined efficiency
    notes           = 'Motor + 40A ESC combo',
)

# ══════════════════════════════════════════════════════════════════════════════

print('Equipment preview:')
for k, v in NEW_EQUIP.items():
    print(f'  {k:<22}: {v}')

Equipment preview:
  equip_id              : MOT_MY_MOTOR_KV400
  category              : Propulsion
  manufacturer          : T-Motor
  model                 : MN601-S KV400
  nom_voltage_v         : 22.2
  nom_current_a         : 20.0
  idle_power_w          : 5.0
  hover_power_w         : 180.0
  climb_power_w         : 230.0
  cruise_power_w        : 130.0
  max_power_w           : 400.0
  weight_g              : 185.0
  efficiency_pct        : 85.0
  notes                 : Motor + 40A ESC combo


In [6]:
# ── Set SAVE_EQUIP = True to write to battery_db.xlsx ────────────────────────
SAVE_EQUIP = False

if SAVE_EQUIP:
    from openpyxl import load_workbook
    wb   = load_workbook(DB_PATH)
    ws   = wb['Equipment_DB']
    # Find header row to get column order
    headers = [c.value for c in ws[3]]
    row = [NEW_EQUIP.get(str(h).lower().replace(' ', '_'), None) for h in headers]
    # Fallback: write by known column names
    col_map = {
        'equip_id': NEW_EQUIP['equip_id'],
        'category': NEW_EQUIP['category'],
        'manufacturer': NEW_EQUIP['manufacturer'],
        'model': NEW_EQUIP['model'],
        'nom_voltage_v': NEW_EQUIP['nom_voltage_v'],
        'nom_current_a': NEW_EQUIP['nom_current_a'],
        'idle_power_w': NEW_EQUIP['idle_power_w'],
        'hover_power_w': NEW_EQUIP['hover_power_w'],
        'climb_power_w': NEW_EQUIP['climb_power_w'],
        'cruise_power_w': NEW_EQUIP['cruise_power_w'],
        'max_power_w': NEW_EQUIP['max_power_w'],
        'weight_g': NEW_EQUIP['weight_g'],
        'efficiency_pct': NEW_EQUIP['efficiency_pct'],
        'notes': NEW_EQUIP['notes'],
        'active': 1,
    }
    new_row = [col_map.get(str(h).lower() if h else '', None) for h in headers]
    ws.append(new_row)
    wb.save(DB_PATH)
    print(f'Saved {NEW_EQUIP["equip_id"]} to Equipment_DB in {DB_PATH}')
    print('Reload the database in other notebooks with: db.reload()')
else:
    print('Preview only — set SAVE_EQUIP = True to write to the database.')

Preview only — set SAVE_EQUIP = True to write to the database.


---
## 3 · Scrape Battery Data from a URL

Paste a product page URL (e.g. manufacturer or retailer) and the scraper will extract
voltage, capacity, energy, weight, and IR values from the page text.

In [7]:
# Check / install scraping dependencies
try:
    import requests
    from bs4 import BeautifulSoup
    print('requests + beautifulsoup4 available')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4'], check=True)
    import requests
    from bs4 import BeautifulSoup
    print('Installed requests + beautifulsoup4')

requests + beautifulsoup4 available


In [8]:
# ── Paste the URL of a battery product page ───────────────────────────────────
SCRAPE_URL = 'https://example.com/battery-product'   # <── change this

# ─────────────────────────────────────────────────────────────────────────────

def _extract_number(text, pattern, group=1):
    """Extract first number matching a regex pattern."""
    m = re.search(pattern, text, re.IGNORECASE)
    if m:
        try: return float(m.group(group).replace(',', '.'))
        except: return None
    return None

def scrape_battery_page(url):
    headers = {'User-Agent': 'Mozilla/5.0 (compatible; BattSim/1.0)'}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f'Error fetching URL: {e}')
        return {}
    soup = BeautifulSoup(resp.text, 'html.parser')
    # Remove script/style
    for tag in soup(['script', 'style', 'nav', 'footer']): tag.decompose()
    text = soup.get_text(separator=' ', strip=True)

    result = {
        'url':           url,
        'page_title':    soup.title.string.strip() if soup.title else '',
        # Voltage: "22.2V" or "6S (22.2V)" or "Voltage: 22.2V"
        'voltage_v':     _extract_number(text, r'(?:voltage|nominal)[^\d]{0,20}([\d.]+)\s*[Vv]'),
        'cells_s':       _extract_number(text, r'(\d)S\b'),
        'cells_p':       _extract_number(text, r'(\d)P\b'),
        # Capacity: "5000mAh" or "5Ah"
        'capacity_mah':  _extract_number(text, r'([\d,]+)\s*mAh'),
        'capacity_ah':   _extract_number(text, r'([\d.]+)\s*Ah(?!\s*per)'),
        # Energy: "111Wh" or "111 Wh"
        'energy_wh':     _extract_number(text, r'([\d.]+)\s*Wh'),
        # Weight: "850g" or "850 g" or "0.85 kg"
        'weight_g':      _extract_number(text, r'([\d.]+)\s*g(?:rams?)?\b'),
        'weight_kg':     _extract_number(text, r'([\d.]+)\s*kg\b'),
        # IR: "6mΩ" or "6 mOhm"
        'ir_mohm':       _extract_number(text, r'([\d.]+)\s*m[Ωω]|([\d.]+)\s*mOhm'),
        # C-rate: "25C" continuous discharge
        'c_rate_cont':   _extract_number(text, r'(\d+)C\s+(?:continuous|cont|discharge)'),
        # Dimensions
        'dims_mm':       re.search(r'(\d+)\s*[x×]\s*(\d+)\s*[x×]\s*(\d+)\s*mm', text),
    }

    # Normalise capacity
    if result['capacity_mah'] and not result['capacity_ah']:
        result['capacity_ah'] = result['capacity_mah'] / 1000
    if result['weight_kg'] and not result['weight_g']:
        result['weight_g'] = result['weight_kg'] * 1000
    if result['dims_mm']:
        m = result['dims_mm']
        result['dims_mm'] = f'{m.group(1)}×{m.group(2)}×{m.group(3)} mm'

    return result


scraped = scrape_battery_page(SCRAPE_URL)
print('Scraped data:')
print(f'  Page          : {scraped.get("page_title")}')
print(f'  URL           : {scraped.get("url")}')
print(f'  Voltage       : {scraped.get("voltage_v")} V')
print(f'  Config        : {scraped.get("cells_s")}S {scraped.get("cells_p")}P')
print(f'  Capacity      : {scraped.get("capacity_ah")} Ah')
print(f'  Energy        : {scraped.get("energy_wh")} Wh')
print(f'  Weight        : {scraped.get("weight_g")} g')
print(f'  IR            : {scraped.get("ir_mohm")} mΩ')
print(f'  Max C-rate    : {scraped.get("c_rate_cont")}C')
print(f'  Dimensions    : {scraped.get("dims_mm")}')
print()
print('Review the values above, copy them into Section 1 form, then save.')

Error fetching URL: HTTPSConnectionPool(host='example.com', port=443): Max retries exceeded with url: /battery-product (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
Scraped data:
  Page          : None
  URL           : None
  Voltage       : None V
  Config        : NoneS NoneP
  Capacity      : None Ah
  Energy        : None Wh
  Weight        : None g
  IR            : None mΩ
  Max C-rate    : NoneC
  Dimensions    : None

Review the values above, copy them into Section 1 form, then save.


---
## 4 · Extract Data from a PDF Datasheet

Point to a PDF file and the extractor will pull out key battery parameters.

In [9]:
# Check / install PDF dependency
try:
    import pdfplumber
    print('pdfplumber available')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'pdfplumber'], check=True)
    import pdfplumber
    print('Installed pdfplumber')

Installed pdfplumber


In [10]:
import pdfplumber

# ── Path to your PDF datasheet ────────────────────────────────────────────────
PDF_PATH = '../logs/battery_datasheet.pdf'   # <── change this

# ─────────────────────────────────────────────────────────────────────────────

def extract_battery_pdf(pdf_path):
    if not os.path.exists(pdf_path):
        print(f'File not found: {pdf_path}')
        return {}, ''

    text_pages = []
    tables_found = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text_pages.append(page.extract_text() or '')
            tbl = page.extract_table()
            if tbl:
                tables_found.append(pd.DataFrame(tbl[1:], columns=tbl[0]))

    full_text = ' '.join(text_pages)

    result = {
        'voltage_v':    _extract_number(full_text, r'(?:nominal voltage|rated voltage)[^\d]{0,30}([\d.]+)\s*V'),
        'cells_s':      _extract_number(full_text, r'(\d)S\b'),
        'cells_p':      _extract_number(full_text, r'(\d)P\b'),
        'capacity_mah': _extract_number(full_text, r'([\d,]+)\s*mAh'),
        'capacity_ah':  _extract_number(full_text, r'([\d.]+)\s*Ah'),
        'energy_wh':    _extract_number(full_text, r'([\d.]+)\s*Wh'),
        'weight_g':     _extract_number(full_text, r'([\d.]+)\s*g(?:rams?)?\b'),
        'ir_mohm':      _extract_number(full_text, r'(?:internal resistance|IR)[^\d]{0,30}([\d.]+)\s*m'),
        'max_c_rate':   _extract_number(full_text, r'(\d+)C\s*(?:max|maximum|burst|peak)'),
        'cont_c_rate':  _extract_number(full_text, r'(\d+)C\s*(?:continuous|cont|discharge)'),
        'temp_min_c':   _extract_number(full_text, r'(?:min|minimum)\s*(?:temp|temperature)[^\d-]{0,20}(-?\d+)'),
        'temp_max_c':   _extract_number(full_text, r'(?:max|maximum)\s*(?:temp|temperature)[^\d]{0,20}(\d+)'),
        'cycle_life':   _extract_number(full_text, r'(\d+)\s*cycles?'),
    }
    if result['capacity_mah'] and not result['capacity_ah']:
        result['capacity_ah'] = result['capacity_mah'] / 1000

    return result, tables_found


pdf_data, pdf_tables = extract_battery_pdf(PDF_PATH)

if pdf_data:
    print('Extracted from PDF:')
    for k, v in pdf_data.items():
        if v is not None:
            print(f'  {k:<20}: {v}')

    if pdf_tables:
        print(f'\nFound {len(pdf_tables)} table(s) in PDF:')
        for i, tbl in enumerate(pdf_tables):
            print(f'\nTable {i+1}:')
            print(tbl.to_string(index=False))

    print('\nReview the values above, copy them into Section 1 form, then save.')

File not found: ../logs/battery_datasheet.pdf


---
## 5 · Bulk Scrape from Tattu / Grepow

Automatically fetch and parse all UAV battery listings from manufacturer websites.
Results are shown as a DataFrame before anything is saved — review and confirm first.

**Run the cells below in order:**
1. Configure options
2. Run the scraper (takes 1-3 minutes — polite 1.5 s delay between requests)
3. Review the preview table
4. Save to database if the data looks good

In [11]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ── Configure the bulk scraper ─────────────────────────────────────────────────
SCRAPE_BRAND    = 'all'     # 'tattu' | 'grepow' | 'all'
MAX_PER_BRAND   = 50        # max product pages to visit per brand
VERBOSE         = True      # print progress line-by-line

# ─────────────────────────────────────────────────────────────────────────────
print(f'Brand: {SCRAPE_BRAND}  |  Max per brand: {MAX_PER_BRAND}')
print('Ready — run the next cell to start scraping.')

Brand: all  |  Max per brand: 50
Ready — run the next cell to start scraping.


In [12]:
from scripts.scrape_batteries import scrape_tattu, scrape_grepow, scrape_all
import pandas as pd

if SCRAPE_BRAND == 'tattu':
    scraped_rows = scrape_tattu(max_products=MAX_PER_BRAND, verbose=VERBOSE)
elif SCRAPE_BRAND == 'grepow':
    scraped_rows = scrape_grepow(max_products=MAX_PER_BRAND, verbose=VERBOSE)
else:
    scraped_rows = scrape_all(max_per_brand=MAX_PER_BRAND, verbose=VERBOSE)

print(f'\nTotal packs scraped: {len(scraped_rows)}')

── Tattu (genstattu.com) ────────────────────────
  Found 50 product URLs
  [1/50] https://genstattu.com/tattu-ultra-high-voltage-41000mah-5c-23-7v-6s1p-g-tech-lipo-battery-with-qs8-plug/
    ✓ Tattu Ultra High Voltage 41000mAh 23.7V 5C 6S1P G-Tech Lipo Battery Pack with XT90S Plug  41000mAh 14S  3790.0g
  [2/50] https://genstattu.com/tattu-4-0-35000mah-68-4v-18s1p-35c-high-voltage-lipo-smart-battery-pack-with-aces-plug-or-molex-plug/
    ✓ Tattu 4.0 35000mAh 68.4V 35C 18S1P Lipo Smart Battery Pack with ACES Plug  35000mAh 14S  350.0g
  [3/50] https://genstattu.com/tattu-semi-solid-state-350wh-kg-33000mah-10c-51-8v-14s1p-lipo-battery-pack-with-as150u-f-plug/
    ✓ Tattu Semi-solid State 350Wh/kg 33000mAh 10C 51.8V 14S1P G-Tech Lipo Battery Pack with AS150U-F  33000mAh 14S  5486.5g
  [4/50] https://genstattu.com/tattu-semi-solid-state-350wh-kg-33000mah-10c-44-4v-12s1p-lipo-battery-pack-with-as150u-f-plug/
  [5/50] https://genstattu.com/tattu-semi-solid-state-350wh-kg-33000mah-10c-22-2v-

In [13]:
# Preview the scraped results before saving
if not scraped_rows:
    print('No data scraped — check the warnings above.')
else:
    df_scraped = pd.DataFrame(scraped_rows)[[
        'battery_id', 'name', 'cells_series', 'cells_parallel',
        'pack_capacity_ah', 'pack_voltage_nom', 'pack_energy_wh',
        'cont_c_rate', 'max_cont_discharge_a',
        'pack_weight_g', 'specific_energy_wh_kg',
    ]]
    df_scraped['pack_capacity_ah'] = (df_scraped['pack_capacity_ah'] * 1000).round(0).astype(int)
    df_scraped = df_scraped.rename(columns={'pack_capacity_ah': 'capacity_mah'})
    pd.set_option('display.max_rows', 100)
    pd.set_option('display.max_colwidth', 40)
    pd.set_option('display.width', 160)
    print(df_scraped.to_string(index=False))
    print(f'\n{len(df_scraped)} packs ready to save.')
    print('Set SAVE_SCRAPED = True in the next cell and re-run to write to battery_db.xlsx.')

                                        battery_id                                                                                             name  cells_series  cells_parallel  capacity_mah  pack_voltage_nom  pack_energy_wh  cont_c_rate  max_cont_discharge_a  pack_weight_g  specific_energy_wh_kg
TATTU_TATTU_ULTRA_HIGH_VOLTAGE_41000MAH_5C_23_7V_6         Tattu Ultra High Voltage 41000mAh 23.7V 5C 6S1P G-Tech Lipo Battery Pack with XT90S Plug            14               7         41000              51.8          2123.8          5.0                 205.0         3790.0                  560.4
TATTU_TATTU_4_0_35000MAH_68_4V_18S1P_35C_HIGH_VOLT                        Tattu 4.0 35000mAh 68.4V 35C 18S1P Lipo Smart Battery Pack with ACES Plug            14               1         35000              51.8          1813.0         35.0                1225.0          350.0                 5180.0
TATTU_TATTU_SEMI_SOLID_STATE_350WH_KG_33000MAH_10C  Tattu Semi-solid State 350Wh/kg 33000mAh 10C 51.8V 

In [14]:
# ── Set SAVE_SCRAPED = True to write results to battery_db.xlsx ───────────────
SAVE_SCRAPED   = False      # <── change to True when you are happy with the preview
SKIP_EXISTING  = True       # skip battery_ids already in the database
DB_PATH_SCRAPE = '../battery_db.xlsx'

# ─────────────────────────────────────────────────────────────────────────────
if SAVE_SCRAPED and scraped_rows:
    from scripts.scrape_batteries import save_to_db
    n = save_to_db(scraped_rows, db_path=DB_PATH_SCRAPE,
                   skip_existing=SKIP_EXISTING, verbose=True)
    print(f'\nWrote {n} new pack(s) to {DB_PATH_SCRAPE}')
    print('Reload the BatteryDatabase in other notebooks to see the new packs.')
elif not SAVE_SCRAPED:
    print('SAVE_SCRAPED is False — nothing written.')
    print('Set SAVE_SCRAPED = True and re-run this cell to save.')
else:
    print('No data to save.')

SAVE_SCRAPED is False — nothing written.
Set SAVE_SCRAPED = True and re-run this cell to save.
